# Nucleosome (TIS-DNA + SOP-histone, explicit ions) — analysis notebook

**Design rules**
1. Every analysis block is followed by an *Experimental comparison* note saying what it is compared to and what the caveats are.
2. Native contacts are defined from the **original crystal structure (2CV5)** after converting it to your CG representation (DNA → P/S/B beads as in `AA2TIS.py`, histones → CA beads as in `AA2CG.py`). Because 2CV5 does not contain the tails you added, only the resolved histone **core** is used, automatically.
3. New per-frame analyses are added by writing one function with the `@observable()` decorator (see the last cell). Nothing else has to change.

**Run order:** config → reference → mapping → observables → run → analysis cells.

In [1]:
import os, json, pickle, warnings
from collections import defaultdict
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
try:
    import MDAnalysis as mda
except ImportError:
    mda = None   # only needed for the trajectory-reading cells
warnings.filterwarnings("ignore", category=DeprecationWarning)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

In [2]:
# ───────────────────────────── CONFIG ─────────────────────────────
AA_PDB          = "2CV5.pdb"                 # original all-atom structure (path on your machine)
DNA_CHAINS      = ["I", "J"]                 # 2CV5: 146-bp palindromic DNA
HISTONE_CHAINS  = list("ABCDEFGH")

# native protein(core)–DNA contacts, defined on the CG structure
R_NAT           = 8.0      # Å, CA–DNA-bead distance in the crystal CG model (check sensitivity cell)
NAT_DNA_BEADS   = ("P", "S", "B")
GAMMA           = 1.2      # contact counted as formed if d < GAMMA * d0
BOUND_FRAC      = 0.5      # a bp counts as "bound" if >= this fraction of its native contacts are formed

TEMP            = 300.0    # K (use the Temp value from your input json)
KB              = 0.0019872041   # kcal/mol/K
KT              = KB * TEMP

# sampling
STRIDE          = 10       # analyse every STRIDE-th frame
ION_EVERY       = 10       # run the (expensive) ion analysis every ION_EVERY-th analysed frame
EQUIL_FRAC      = 0.2      # discard first 20 % of each replica
FRAMES_TO_TIME  = None     # e.g. 1e-3 to convert frame index to some time unit; None -> frames

# ions
ION_RESNAMES    = {"K": ("K",), "Cl": ("CL",), "Mg": ("MG",)}
ION_Q           = {"K": +1, "Cl": -1, "Mg": +2}
ION_SURFACE     = "complex"   # "complex" (DNA+all histone beads) or "dna"; distance-to-nearest-bead shells
R_MAX, DR       = 100.0, 2.0  # Å, shell range / width for excess-ion profiles (R_MAX is also the 'bulk' boundary)
R_PLATEAU       = 50.0     # Å, where excess counts are read off (should be on the plateau)
R_LOC           = 10.0     # Å, sphere around each phosphate for local ion concentration
R_COND          = 7.0      # Å, "condensed" ions = within this distance of any phosphate
N_MC, N_VOL_FRAMES = 300000, 8   # Monte-Carlo points / frames used for shell volumes

FRET_PAIRS      = [(0, 145), (10, 135)]   # (bp_i, bp_j), 0-based bp index along the DNA; edit to mimic dye positions

# data: one entry per condition (use this for the Mg scan); each replica = list of DCDs concatenated in order
BASE = "/data/cg_dna_histone/combined/submission_3"
def _dcds(sim, run):
    # ASSUMPTION: 4_day_run_2 continues 4_day_run_1 for the same sim/run. Change if that is not true.
    return [f"{BASE}/{s}/nucleosome_sim_{sim}/run{run}/dna_traj.dcd" for s in ("4_day_run_1", "4_day_run_2")]
CONDITIONS = {
    "baseline": dict(
        mg_mM=None,                                   # fill in for the Mg scan
        top=f"{BASE}/4_day_run_1/nucleosome_sim_1/run1/dna_initial.pdb",
        replicas={f"sim{s}_run{r}": _dcds(s, r) for s in (1, 2, 3, 4) for r in (1, 2)},
    ),
}
CACHE_DIR, FORCE_RECOMPUTE = "analysis_cache", False
PROT_OFFSET_OVERRIDE = {}   # e.g. {"D": 4}: sim_resid + offset = 2CV5 resid, only if auto-detection fails

## 1. Reference: 2CV5 → CG (same grouping as `AA2TIS.py` / `AA2CG.py`) and native contacts

In [3]:
MASS  = {"C": 12.011, "N": 14.007, "O": 15.999, "P": 30.974}
PHOS  = {"P", "OP1", "OP2", "O1P", "O2P", "O5'"}
SUGAR = {"C1'", "C2'", "C3'", "C4'", "C5'", "O4'", "O3'", "O2'"}
RES1  = {"DA": "A", "DT": "T", "DG": "G", "DC": "C", "A": "A", "T": "T", "G": "G", "C": "C"}

def _elem(n):
    n = n.strip()
    if n.startswith("P"): return "P"
    for ch in n:
        if ch.isalpha(): return ch
    return "C"

def _com(atoms):
    m = np.array([MASS.get(_elem(k), 12.0) for k in atoms]); x = np.array(list(atoms.values()))
    return (m[:, None] * x).sum(0) / m.sum()

def build_reference(pdb, dna_chains, histone_chains):
    nuc = {c: {} for c in dna_chains}; ca = []; mn = []
    for line in open(pdb):
        rec = line[:6].strip()
        if rec not in ("ATOM", "HETATM"): continue
        if line[16] not in (" ", "A"): continue
        name, res, ch = line[12:16].strip(), line[17:20].strip(), line[21]
        resid = int(line[22:26]); xyz = np.array([float(line[30:38]), float(line[38:46]), float(line[46:54])])
        if ch in dna_chains and res in RES1:
            nuc[ch].setdefault(resid, {"base": RES1[res], "atoms": {}})["atoms"][name] = xyz
        elif rec == "ATOM" and ch in histone_chains and name == "CA":
            ca.append(dict(chain=ch, resid=resid, resname=res, xyz=xyz))
        elif rec == "HETATM" and res == "MN":
            mn.append(dict(xyz=xyz))
    beads = []
    for s, ch in enumerate(dna_chains):
        for k, resid in enumerate(sorted(nuc[ch])):
            r = nuc[ch][resid]; a = r["atoms"]
            ph = {n: c for n, c in a.items() if n in PHOS}
            su = {n: c for n, c in a.items() if n in SUGAR}
            ba = {n: c for n, c in a.items() if n not in PHOS and n not in SUGAR}
            for bead, grp in (("P", ph if len(ph) >= 2 else None), ("S", su), ("B", ba)):
                if grp: beads.append(dict(strand=s, nt=k, chain=ch, resid=resid, bead=bead, base=r["base"], xyz=_com(grp)))
    L = max(b["nt"] for b in beads if b["strand"] == 0) + 1
    assert max(b["nt"] for b in beads if b["strand"] == 1) + 1 == L, "strands differ in length"
    for b in beads: b["bp"] = b["nt"] if b["strand"] == 0 else L - 1 - b["nt"]   # antiparallel pairing
    # which nucleotide coordinates the crystallographic Mn2+ ions (for the Mg-site comparison)
    mn_sites = []
    for m in mn:
        best = None
        for s, ch in enumerate(dna_chains):
            for k, resid in enumerate(sorted(nuc[ch])):
                for an, c in nuc[ch][resid]["atoms"].items():
                    d = np.linalg.norm(c - m["xyz"])
                    if d < 3.2 and (best is None or d < best[0]): best = (d, s, k, an)
        if best: mn_sites.append(dict(strand=best[1], nt=best[2], bp=best[2] if best[1] == 0 else L - 1 - best[2], atom=best[3], d=best[0]))
    return dict(dna=beads, dna_xyz=np.array([b["xyz"] for b in beads]), ca=ca,
                ca_xyz=np.array([c["xyz"] for c in ca]), L=L, mn_sites=mn_sites)

def native_contacts(ref, r_nat, beads=NAT_DNA_BEADS):
    sel = np.array([i for i, b in enumerate(ref["dna"]) if b["bead"] in beads])
    sdm = cKDTree(ref["ca_xyz"]).sparse_distance_matrix(cKDTree(ref["dna_xyz"][sel]), r_nat, output_type="ndarray")
    return dict(i=sdm["i"].astype(int), j=sel[sdm["j"].astype(int)], d0=sdm["v"].astype(float))

if os.path.exists(AA_PDB):
    REF = build_reference(AA_PDB, DNA_CHAINS, HISTONE_CHAINS)
    NAT = native_contacts(REF, R_NAT)
    nb = pd.Series([REF["dna"][j]["bead"] for j in NAT["j"]]).value_counts()
    print(f"DNA: {REF['L']} bp, {len(REF['dna'])} beads | histone-core CA beads (tails absent in 2CV5): {len(REF['ca'])}")
    print(f"Native core–DNA contacts (R_NAT={R_NAT} Å): {len(NAT['i'])}  by DNA bead type: {nb.to_dict()}")
    print("Crystallographic Mn2+ sites (nt coordinated):", [(m['bp'], m['atom'], round(m['d'], 2)) for m in REF["mn_sites"]])
else:
    print(f"{AA_PDB} not found — set AA_PDB")

DNA: 146 bp, 874 beads | histone-core CA beads (tails absent in 2CV5): 761
Native core–DNA contacts (R_NAT=8.0 Å): 519  by DNA bead type: {'P': 302, 'S': 214, 'B': 3}
Crystallographic Mn2+ sites (nt coordinated): [(67, 'O6', np.float64(2.38)), (120, 'N7', np.float64(2.53)), (133, 'N7', np.float64(2.89)), (107, 'N7', np.float64(2.46)), (25, 'N7', np.float64(2.25)), (75, 'N7', np.float64(2.6)), (12, 'N7', np.float64(2.74))]


In [4]:
# sensitivity of the native-contact definition (report the one you use; check that conclusions do not hinge on it)
if "REF" in globals():
    for rc in (6, 7, 8, 9, 10, 12):
        n = native_contacts(REF, rc)
        print(f"R_NAT={rc:>4} Å -> {len(n['i']):5d} contacts, {len(np.unique(n['j'])):4d} DNA beads, "
              f"{len(np.unique([REF['dna'][j]['bp'] for j in n['j']])):4d} bp touched")

R_NAT=   6 Å ->   160 contacts,   83 DNA beads,   61 bp touched
R_NAT=   7 Å ->   295 contacts,  122 DNA beads,   82 bp touched
R_NAT=   8 Å ->   519 contacts,  171 DNA beads,   96 bp touched
R_NAT=   9 Å ->   842 contacts,  221 DNA beads,  113 bp touched
R_NAT=  10 Å ->  1296 contacts,  282 DNA beads,  125 bp touched
R_NAT=  12 Å ->  2757 contacts,  407 DNA beads,  136 bp touched


## 2. Map reference → simulation topology
Matching is done by **identity, not by index**: DNA beads by (strand, nucleotide order, bead type); histone core beads by (chain, residue number) with an automatic residue-number offset search (checked against residue names), so the extra tail residues in your model are simply ignored. Read the printed diagnostics — if anything is unmatched it stops with an explanation.

In [5]:
AA3 = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS HID HIE HIP ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL".split())

def mic(d, L): return d if L is None else d - L * np.round(d / L)
def wrap(x, L): x = np.mod(x, L); return np.where(x >= L, 0.0, x)

def _segments(chains, resids):
    seg = np.zeros(len(resids), int)
    for k in range(1, len(resids)):
        seg[k] = seg[k-1] + (chains[k] != chains[k-1] or resids[k] < resids[k-1])
    return seg

def _bead_type(name):
    n = name.strip().upper()
    return "P" if n == "P" else ("S" if n == "S" else "B")

def make_static(info, ref, nat):
    names, resn, resid, chain = info["names"], info["resnames"], info["resids"], info["chains"]
    up = np.char.upper(resn.astype(str))
    dna_mask = np.isin(up, ["DA", "DT", "DG", "DC"])
    if not dna_mask.any(): dna_mask = np.isin(names, ["P", "S", "B"])
    prot_mask = np.isin(up, list(AA3)) | np.isin(names, ["BB"])
    dna_idx, prot_idx = np.where(dna_mask)[0], np.where(prot_mask)[0]
    L = ref["L"]

    # ---- DNA keys (strand, nt, bead) ----
    strand = np.zeros(len(dna_idx), int); nt = np.zeros(len(dna_idx), int)
    s = k = 0
    for q in range(1, len(dna_idx)):
        a, b = dna_idx[q-1], dna_idx[q]
        if chain[a] != chain[b] or resid[b] < resid[a]: s += 1; k = 0
        elif resid[b] != resid[a]: k += 1
        strand[q], nt[q] = s, k
    if strand.max() == 0:                      # single chain label/numbering: split in half
        n_nt = nt.max() + 1; assert n_nt % 2 == 0, "cannot split DNA into two strands"
        strand = (nt >= n_nt // 2).astype(int); nt = nt - strand * (n_nt // 2)
    assert nt[strand == 0].max() + 1 == L and nt[strand == 1].max() + 1 == L, \
        f"DNA length mismatch: sim has {nt[strand==0].max()+1}/{nt[strand==1].max()+1} nt per strand, 2CV5 has {L}"
    key2local = {(strand[q], nt[q], _bead_type(names[dna_idx[q]])): q for q in range(len(dna_idx))}
    ref2dna = np.array([key2local.get((b["strand"], b["nt"], b["bead"]), -1) for b in ref["dna"]])
    print(f"DNA: {len(dna_idx)} sim beads, {L} nt/strand; reference beads missing in sim: {(ref2dna < 0).sum()}")

    # ---- histone core mapping ----
    pc, pr, pn = chain[prot_idx], resid[prot_idx], up[prot_idx]
    seg = _segments(pc, pr); nseg = seg.max() + 1
    seg_chain = [pc[np.where(seg == q)[0][0]] for q in range(nseg)]
    if sorted(seg_chain) == sorted(HISTONE_CHAINS): ref_of = {q: seg_chain[q] for q in range(nseg)}
    elif nseg == len(HISTONE_CHAINS): ref_of = {q: HISTONE_CHAINS[q] for q in range(nseg)}
    else: raise ValueError(f"found {nseg} protein segments ({seg_chain}); expected {len(HISTONE_CHAINS)} — set up chain mapping")
    refres = {c: {x["resid"]: x["resname"].upper() for x in ref["ca"] if x["chain"] == c} for c in HISTONE_CHAINS}
    lookup, rows = {}, []
    for q in range(nseg):
        c = ref_of[q]; m = np.where(seg == q)[0]
        best = max(range(-10, 11), key=lambda o: sum(refres[c].get(r + o) == n for r, n in zip(pr[m], pn[m])))
        off = PROT_OFFSET_OVERRIDE.get(c, best)
        hit = sum(refres[c].get(r + off) == n for r, n in zip(pr[m], pn[m]))
        rows.append((c, seg_chain[q], len(m), off, hit, len(refres[c])))
        for a, r in zip(prot_idx[m], pr[m]): lookup[(c, r + off)] = a
    print(pd.DataFrame(rows, columns=["2CV5 chain", "sim chain", "sim beads", "resid offset", "name matches", "2CV5 core res"]).to_string(index=False))
    ref2sim_ca = np.array([lookup.get((x["chain"], x["resid"]), -1) for x in ref["ca"]])
    print(f"core CA beads of 2CV5 found in sim: {(ref2sim_ca>=0).sum()}/{len(ref2sim_ca)}  "
          f"(sim histone beads total {len(prot_idx)} -> {len(prot_idx)-(ref2sim_ca>=0).sum()} extra = tails/other, ignored)")

    # ---- native contacts restricted to mapped beads ----
    core_valid = np.where(ref2sim_ca >= 0)[0]
    ref2core = -np.ones(len(ref2sim_ca), int); ref2core[core_valid] = np.arange(len(core_valid))
    ok = (ref2core[nat["i"]] >= 0) & (ref2dna[nat["j"]] >= 0)
    if (~ok).any(): print(f"WARNING: {(~ok).sum()} native contacts dropped (unmapped beads)")
    ci, dj, d0, jref = ref2core[nat["i"][ok]], ref2dna[nat["j"][ok]], nat["d0"][ok], nat["j"][ok]
    c_bp = np.array([ref["dna"][j]["bp"] for j in jref])
    S = dict(L=L, dna_idx=dna_idx, prot_idx=prot_idx, core_sim=ref2sim_ca[core_valid],
             core_ref_xyz=ref["ca_xyz"][core_valid], c_ci=ci, c_dj=dj, c_d0=d0, c_bp=c_bp,
             bp_native=np.bincount(c_bp, minlength=L).astype(float))
    # base beads per bp (for bp centres)
    B = lambda st, n: key2local[(st, n, "B")]
    S["bp_B1"] = np.array([B(0, i) for i in range(L)]); S["bp_B2"] = np.array([B(1, L-1-i) for i in range(L)])
    pl = [(q, strand[q] * L + nt[q]) for q in range(len(dna_idx)) if _bead_type(names[dna_idx[q]]) == "P"]
    S["P_local"] = np.array([a for a, _ in pl]); S["P_nt"] = np.array([b for _, b in pl])
    S["n_phosphate"] = len(pl)
    # ions
    S["ion_idx"] = {}
    for sp, rn in ION_RESNAMES.items():
        S["ion_idx"][sp] = np.where(np.isin(up, [x.upper() for x in rn]))[0]
    print("ions:", {k: len(v) for k, v in S["ion_idx"].items()}, "| phosphates:", S["n_phosphate"])
    # reference geometry (superhelix axis, centre, handedness) from the crystal bp centres
    bpref = np.zeros((L, 3)); cnt = np.zeros(L)
    for b in ref["dna"]:
        if b["bead"] == "B": bpref[b["bp"]] += b["xyz"]; cnt[b["bp"]] += 1
    bpref /= cnt[:, None]
    o = bpref.mean(0); w, v = np.linalg.eigh(np.cov((bpref - o).T)); axis = v[:, 0]
    e1 = v[:, 2]; e2 = np.cross(axis, e1); basis = np.stack([e1, e2, axis], 1)
    q = (bpref - o) @ basis
    A = np.c_[q[:, 0], q[:, 1], np.ones(L)]; sol = np.linalg.lstsq(A, -(q[:, 0]**2 + q[:, 1]**2), rcond=None)[0]
    S.update(g_o=o, g_basis=basis, g_cx=-sol[0]/2, g_cy=-sol[1]/2)
    ph = np.unwrap(np.arctan2(q[:, 1] - S["g_cy"], q[:, 0] - S["g_cx"]))
    S["g_sign"] = 1.0 if ph[-1] > ph[0] else -1.0
    S["bp_ref"] = bpref
    return S

def info_from_universe(u):
    a = u.atoms
    ch = a.chainIDs if hasattr(a, "chainIDs") else a.segids
    return dict(names=np.array(a.names), resnames=np.array(a.resnames), resids=np.array(a.resids), chains=np.array(ch))

## 3. Observables (per-frame analyses) and the engine
Each observable gets `ctx` (coordinates already made whole across the periodic box: `core`, `dna`, `prot`, `pos`, `box`, `res` = results of earlier observables in this frame) and the static maps `S`, and returns a dict of floats/arrays.

In [6]:
OBS = []
def observable(every=1):
    def deco(fn): OBS.append((fn, every)); return fn
    return deco

def kabsch(P, Q):
    Pc, Qc = P.mean(0), Q.mean(0)
    U, _, Vt = np.linalg.svd((P - Pc).T @ (Q - Qc))
    D = np.diag([1, 1, np.sign(np.linalg.det(Vt.T @ U.T))])
    R = Vt.T @ D @ U.T
    return R, Qc - R @ Pc

def bp_centres(dna, S): return 0.5 * (dna[S["bp_B1"]] + dna[S["bp_B2"]])

def geom_of_path(bpf, S, a, b):
    """wrapping angle (deg), mean radius (Å), pitch (Å) of bp centres a..b already in the reference frame."""
    q = (bpf - S["g_o"]) @ S["g_basis"]
    x, y, z = q[:, 0] - S["g_cx"], q[:, 1] - S["g_cy"], q[:, 2]
    phi = S["g_sign"] * np.unwrap(np.arctan2(y, x)); r = np.hypot(x, y)
    if b - a < 10: return np.nan, np.nan, np.nan
    sl = slice(a, b + 1)
    ang = abs(phi[b] - phi[a]) * 180 / np.pi
    pitch = abs(np.polyfit(phi[sl], z[sl], 1)[0]) * 2 * np.pi
    return ang, r[sl].mean(), pitch

@observable(1)
def obs_contacts(ctx, S):
    L = S["L"]
    d = np.linalg.norm(ctx["dna"][S["c_dj"]] - ctx["core"][S["c_ci"]], axis=1)
    formed = d < GAMMA * S["c_d0"]
    bpf = np.bincount(S["c_bp"], weights=formed, minlength=L)
    frac = np.full(L, np.nan); m = S["bp_native"] > 0; frac[m] = bpf[m] / S["bp_native"][m]
    idx = np.where(frac >= BOUND_FRAC)[0]
    u5, u3 = (L, L) if len(idx) == 0 else (idx[0], L - 1 - idx[-1])
    return dict(Q=formed.mean(), bp_frac=frac.astype(np.float32), u5=u5, u3=u3, n_wrap=max(L - u5 - u3, 0))

@observable(1)
def obs_geometry(ctx, S):
    L = S["L"]; bp = bp_centres(ctx["dna"], S)
    R, t = kabsch(ctx["core"], S["core_ref_xyz"]); bpf = bp @ R.T + t
    a, b = ctx["res"]["u5"], L - 1 - ctx["res"]["u3"]
    ang, rad, pit = geom_of_path(bpf, S, a, b) if b > a else (np.nan,) * 3
    out = dict(wrap_angle=ang, radius=rad, pitch=pit, end_dist=np.linalg.norm(bp[0] - bp[-1]),
               core_rmsd=np.sqrt(((ctx["core"] @ R.T + t - S["core_ref_xyz"])**2).sum(1).mean()))
    out["fret_dist"] = np.array([np.linalg.norm(bp[i] - bp[j]) for i, j in FRET_PAIRS])
    return out

EDGES = np.arange(0.0, R_MAX + DR, DR)
def _ion_volumes(surf, L, S):
    if len(S.setdefault("_vol", [])) >= N_VOL_FRAMES: return np.mean(S["_vol"], 0)
    rng = np.random.default_rng(len(S["_vol"]))
    c = surf.mean(0); half = np.abs(surf - c).max() + EDGES[-1] + 1
    assert 2 * half < L.min(), "sampling cube larger than box: reduce R_MAX"
    pts = c + (rng.random((N_MC, 3)) * 2 - 1) * half
    d = cKDTree(wrap(surf, L), boxsize=L).query(wrap(pts, L))[0]
    cum = np.cumsum(np.histogram(d, EDGES)[0]) / N_MC * (2 * half) ** 3
    S["_vol"].append(cum); return np.mean(S["_vol"], 0)

@observable(ION_EVERY)
def obs_ions(ctx, S):
    L = ctx["box"]
    if L is None: return {}
    surf = np.vstack([ctx["dna"], ctx["prot"]]) if ION_SURFACE == "complex" else ctx["dna"]
    tree = cKDTree(wrap(surf, L), boxsize=L)
    V = _ion_volumes(surf, L, S); Vbulk = np.prod(L) - V[-1]
    out = {}; Pw = wrap(ctx["dna"][S["P_local"]], L); ptree = cKDTree(Pw, boxsize=L)
    for sp, idx in S["ion_idx"].items():
        if len(idx) == 0: continue
        ip = wrap(ctx["pos"][idx], L)
        d = tree.query(ip)[0]
        cum = np.cumsum(np.histogram(d, EDGES)[0]); rho = (len(idx) - cum[-1]) / Vbulk       # bulk density, Å^-3
        out[f"gam_{sp}"] = cum - rho * V                                                       # preferential interaction Γ(r)
        out[f"bulk_M_{sp}"] = rho * 1660.539
        cnt = cKDTree(ip, boxsize=L).query_ball_point(Pw, R_LOC, return_length=True)
        loc = np.full(2 * S["L"], np.nan); loc[S["P_nt"]] = cnt
        out[f"ploc_{sp}"] = loc                                                                # ions within R_LOC of each phosphate
        out[f"ncond_{sp}"] = (ptree.query(ip)[0] < R_COND).sum()
    return out

def build_ctx(pos, box, S):
    core = pos[S["core_sim"]]
    ref0 = core[0]; core = ref0 + mic(core - ref0, box)
    c = core.mean(0)
    return dict(core=core, dna=c + mic(pos[S["dna_idx"]] - c, box), prot=c + mic(pos[S["prot_idx"]] - c, box),
                pos=pos, box=box, res={})

def analyze_frames(frames, S, obs=None):
    obs = obs or OBS; store, fr = defaultdict(list), defaultdict(list)
    for n, (frame, pos, box) in enumerate(frames):
        ctx = build_ctx(pos, box, S)
        for fn, every in obs:
            if n % every: continue
            out = fn(ctx, S); ctx["res"].update(out)
            for k, v in out.items(): store[k].append(v); fr[k].append(frame)
    return {k: np.array(v) for k, v in store.items()}, {k: np.array(v) for k, v in fr.items()}

def md_frames(u, stride):
    for ts in u.trajectory[::stride]:
        box = ts.dimensions[:3].astype(float).copy() if ts.dimensions is not None and ts.dimensions[0] > 0 else None
        yield ts.frame, u.atoms.positions.astype(float), box

## 4. Run all conditions / replicas (results are cached to disk)

In [7]:
RES, FRM, STATIC = {}, {}, {}
os.makedirs(CACHE_DIR, exist_ok=True)
for cond, spec in CONDITIONS.items():
    top = spec["top"]
    if not os.path.exists(top): print(f"[{cond}] topology missing: {top}"); continue
    if mda is None: raise ImportError("MDAnalysis needed to read trajectories")
    u0 = mda.Universe(top)
    S = make_static(info_from_universe(u0), REF, NAT); STATIC[cond] = S
    RES[cond], FRM[cond] = {}, {}
    for rep, files in spec["replicas"].items():
        files = [f for f in files if os.path.exists(f)]
        if not files: print(f"[{cond}/{rep}] no trajectory files found"); continue
        cache = f"{CACHE_DIR}/{cond}__{rep}__s{STRIDE}_r{R_NAT}_g{GAMMA}.pkl"
        if os.path.exists(cache) and not FORCE_RECOMPUTE:
            RES[cond][rep], FRM[cond][rep] = pickle.load(open(cache, "rb")); print(f"[{cond}/{rep}] loaded cache"); continue
        u = mda.Universe(top, files)
        print(f"[{cond}/{rep}] {len(u.trajectory)} frames, analysing every {STRIDE}")
        S["_vol"] = []
        RES[cond][rep], FRM[cond][rep] = analyze_frames(md_frames(u, STRIDE), S)
        pickle.dump((RES[cond][rep], FRM[cond][rep]), open(cache, "wb"))

[baseline] topology missing: /data/cg_dna_histone/combined/submission_3/4_day_run_1/nucleosome_sim_1/run1/dna_initial.pdb


In [8]:
# helpers used by all analysis cells
def prod(cond, rep, name):
    """observable array for one replica with the equilibration part removed"""
    a = RES[cond][rep][name]; return a[int(EQUIL_FRAC * len(a)):]
def all_reps(cond, name): return [prod(cond, r, name) for r in RES[cond]]
def pooled(cond, name): return np.concatenate(all_reps(cond, name), axis=0)
def rep_means(cond, name): return np.array([np.nanmean(a, 0) for a in all_reps(cond, name)])
def mean_sem(x):
    x = np.asarray(x); return np.nanmean(x, 0), np.nanstd(x, 0, ddof=1) / np.sqrt(max(len(x), 2)) if len(x) > 1 else np.zeros_like(np.nanmean(x, 0))
def tax(cond, rep, name):
    """x-axis (frame index, or time if FRAMES_TO_TIME is set) for the full trajectory of one replica"""
    return FRM[cond][rep][name] * (FRAMES_TO_TIME or 1)

## 5. Native contacts Q, wrapped base pairs, per-bp wrapping profile

In [9]:
cond = list(RES)[0]
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for rep in RES[cond]:
    ax[0].plot(tax(cond, rep, "Q"), RES[cond][rep]["Q"], lw=.6, label=rep)
    ax[1].plot(tax(cond, rep, "n_wrap"), RES[cond][rep]["n_wrap"], lw=.6)
ax[0].set(xlabel="frame" if not FRAMES_TO_TIME else "time", ylabel="Q (core–DNA native contacts)"); ax[0].legend(fontsize=6, ncol=2)
ax[1].set(xlabel="frame", ylabel="wrapped bp"); ax[1].axhline(REF["L"], c="k", ls="--", lw=.8)
m, e = mean_sem(np.array([np.nanmean(prod(cond, r, "bp_frac"), 0) for r in RES[cond]]))
ax[2].errorbar(range(REF["L"]), m, e, fmt=".-", ms=3, lw=.6); ax[2].axhline(1, c="k", ls="--", lw=.8)
ax[2].set(xlabel="bp index (0 = 5' end of strand I)", ylabel="fraction of native contacts formed"); plt.tight_layout()
print("per-replica means (production):")
print(pd.DataFrame({r: dict(Q=np.mean(prod(cond, r, "Q")), wrapped_bp=np.mean(prod(cond, r, "n_wrap")),
                            u5=np.mean(prod(cond, r, "u5")), u3=np.mean(prod(cond, r, "u3"))) for r in RES[cond]}).T.round(2))

IndexError: list index out of range

**Experimental comparison — wrapping / breathing**

| Simulation quantity | Experimental counterpart | Caveat |
|---|---|---|
| Reference geometry (wrapped bp ≈ 145-147, ~1.65 turns) | crystal structures: Luger *et al.* 1997 *Nature* 389:251; Davey *et al.* 2002 *JMB* 319:1097; Tsunaka *et al.* 2005 *NAR* 33:3424 (= 2CV5, your source) | The crystal is the target, not an independent measurement of solution behaviour. |
| Per-bp contact occupancy (high in the centre, lower toward the ends) | End-to-dyad gradient of site accessibility: Polach & Widom 1995 *JMB* 254:130; Anderson & Widom 2000 *JMB* 296:979; Li & Widom 2004 *NSMB* 11:763 | Compare the *shape*, not the absolute value. |
| Wrapped bp, unwrapped length at each end (`u5`, `u3`) | Spontaneous end unwrapping (smFRET / restriction accessibility): Li *et al.* 2005 *NSMB* 12:46; Koopmans *et al.* 2007; Gansen *et al.* 2009 *PNAS* | Only ~10–20 bp at each end unwrap appreciably in the experiments; CG time scales are not real time. |

## 6. Site exposure (the direct analogue of restriction-enzyme accessibility) and F(N_wrap)
A site of length *w* bp, starting *s* bp from a DNA end, is "exposed" when the unwrapped length at that end is ≥ *s + w*. Then K(s) = P_exposed / (1 − P_exposed) and ΔG(s) = −kT ln K(s).

In [10]:
def site_exposure(cond, w=6, smax=60):
    s_ = np.arange(0, smax); out = {}
    for end in ("u5", "u3"):
        per_rep = []
        for r in RES[cond]:
            u = prod(cond, r, end); p = np.array([(u >= s + w).mean() for s in s_]); per_rep.append(p)
        out[end] = np.array(per_rep)
    return s_, out

s_, SE = site_exposure(cond)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for end, c in (("u5", "C0"), ("u3", "C1")):
    p = SE[end]; K = p / (1 - p + 1e-12); m, e = mean_sem(K)
    ok = m > 0
    ax[0].semilogy(s_[ok], m[ok], c=c, label=end); ax[0].fill_between(s_[ok], np.maximum(m - e, 1e-6)[ok], (m + e)[ok], color=c, alpha=.2)
ax[0].set(xlabel="site start (bp from DNA end)", ylabel="K_eq,site (6-bp site)"); ax[0].legend()
N = pooled(cond, "n_wrap"); hist, edges = np.histogram(N, bins=np.arange(0, REF["L"] + 4, 3), density=True)
F = -KT * np.log(np.where(hist > 0, hist, np.nan)); ax[1].plot(0.5 * (edges[1:] + edges[:-1]), F - np.nanmin(F), "o-")
ax[1].set(xlabel="wrapped bp", ylabel="F (kcal/mol)"); plt.tight_layout()

KeyError: 'baseline'

**Experimental comparison — site exposure and F(N_wrap)**

* K(s) vs s is the quantity measured by Polach & Widom (1995), Anderson & Widom (2000) and Li & Widom (2004) with restriction enzymes at many positions: K falls steeply (orders of magnitude) from the DNA ends toward the dyad. Look up the table values in those papers and overlay them (ΔG = −kT ln K). Expect agreement in trend; the absolute magnitudes of a CG force field without tails/sequence-specific terms will differ.
* Force spectroscopy gives complementary information on the energetics of unwrapping (Brower-Toland *et al.* 2002 *PNAS* 99:1960; Mihardja *et al.* 2006 *PNAS* 103:15871; Hall *et al.* 2009 *NSMB* 16:124; Ngo *et al.* 2015 *Cell* 160:1135). These need a *pulling* simulation to compare quantitatively; F(N_wrap) from equilibrium runs is only a qualitative cross-check (barrier near the entry/exit region, multi-stage unwrapping).
* Caveat: a 4-day CG run may sample few deep-unwrapping events, so K for large s may be 0 (not measurable) — report which positions are statistically resolved.

## 7. Superhelix geometry and end-to-end distances

In [ ]:
Lbp = REF["L"]
ang_ref, rad_ref, pit_ref = geom_of_path(STATIC[cond]["bp_ref"], STATIC[cond], 0, Lbp - 1)
print(f"CRYSTAL (2CV5 bp centres): wrapping angle = {ang_ref:.0f}° ({ang_ref/360:.2f} turns), radius = {rad_ref:.1f} Å, pitch = {pit_ref:.1f} Å")
fig, ax = plt.subplots(1, 4, figsize=(16, 3.6))
for a, nm, ref_v, lab in zip(ax, ["wrap_angle", "radius", "pitch", "end_dist"], [ang_ref, rad_ref, pit_ref, None],
                             ["wrapping angle (deg)", "superhelix radius (Å)", "pitch (Å)", "bp0–bp(L-1) distance (Å)"]):
    x = pooled(cond, nm); x = x[np.isfinite(x)]
    a.hist(x, bins=60, density=True, alpha=.7); a.set(xlabel=lab)
    if ref_v is not None: a.axvline(ref_v, c="k", ls="--", label="crystal")
ax[0].legend(); plt.tight_layout()
print("core rmsd to crystal (Å):", {r: round(float(np.mean(prod(cond, r, 'core_rmsd'))), 2) for r in RES[cond]})
fd = pooled(cond, "fret_dist")
print("FRET-pair bp distances (Å), mean ± sd:", [(FRET_PAIRS[k], round(fd[:, k].mean(), 1), round(fd[:, k].std(), 1)) for k in range(len(FRET_PAIRS))])

**Experimental comparison — geometry / FRET**

* Wrapping angle ≈ 1.65 left-handed superhelical turns (~590–600°), radius and pitch of the DNA superhelix: crystal values (Luger 1997: pitch ≈ 24 Å, DNA superhelix radius ≈ 42 Å — verify the exact numbers in the paper; the script prints the values from *your* 2CV5 so you compare like with like).
* End-to-end / FRET-pair distances: smFRET on labelled nucleosomes (Li & Widom 2005; Koopmans 2007; Gansen 2009 — pick the paper whose dye positions you want to mimic and set `FRET_PAIRS`). Convert distance to efficiency with the dye's Förster radius **R0** (E = 1/(1+(r/R0)^6)); dye linkers add several Å of uncertainty, so use ensembles/thresholds (open vs closed), not single numbers.

## 8. Ion atmosphere: excess ions, charge neutralisation, Mg²⁺ preference
Γ_i(r) = N_i(<r) − ρ_i,bulk · V(<r), with *r* = distance to the nearest bead of the complex and ρ_bulk measured beyond R_MAX (so box-size depletion is accounted for).

In [ ]:
IONS = [k for k in ION_RESNAMES if f"gam_{k}" in RES[cond][list(RES[cond])[0]]]
rr = EDGES[1:]                       # upper edge of each shell (Å)
G = {k: np.array([np.nanmean(prod(cond, r, f"gam_{k}"), 0) for r in RES[cond]]) for k in IONS}   # reps × bins
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for k in IONS:
    m, e = mean_sem(G[k]); ax[0].plot(rr, m, label=k); ax[0].fill_between(rr, m - e, m + e, alpha=.2)
net = sum(ION_Q[k] * G[k] for k in IONS); m, e = mean_sem(net)
ax[1].plot(rr, m); ax[1].fill_between(rr, m - e, m + e, alpha=.2); ax[1].axhline(0, c="k", lw=.5)
ax[0].set(xlabel="r from complex surface (Å)", ylabel="Γ(r) (excess ions)"); ax[0].legend()
ax[1].set(xlabel="r (Å)", ylabel="Σ q·Γ  (ion-atmosphere charge, e)")
b = {k: np.nanmean([np.nanmean(prod(cond, r, f"bulk_M_{k}")) for r in RES[cond]]) for k in IONS}
print("bulk concentrations (M), measured beyond R_MAX:", {k: round(v, 4) for k, v in b.items()})
ip = int(np.argmin(abs(rr - R_PLATEAU)))
tab = pd.DataFrame({k: dict(zip(["mean", "sem"], [G[k][:, ip].mean(), G[k][:, ip].std(ddof=1) / np.sqrt(len(G[k]))])) for k in IONS}).T
tab.loc["Σ qΓ"] = [net[:, ip].mean(), net[:, ip].std(ddof=1) / np.sqrt(len(net))]
print(f"Excess ions per nucleosome at r = {R_PLATEAU} Å:\n", tab.round(2))
print(f"phosphate charge in model: -{STATIC[cond]['n_phosphate']} e  -> ion atmosphere / phosphate charge = {-tab.loc['Σ qΓ','mean']/STATIC[cond]['n_phosphate']:.2f}")
# share of neutralisation from cation gain vs anion loss
cat = sum(ION_Q[k] * G[k][:, ip].mean() for k in IONS if ION_Q[k] > 0); an = -sum(ION_Q[k] * G[k][:, ip].mean() for k in IONS if ION_Q[k] < 0)
print(f"charge from cation accumulation: {cat:.1f} e ; from anion depletion: {-an:.1f} e")
ax[2].bar(["cations", "anion depletion"], [cat, an]); ax[2].set_ylabel("charge (e)"); plt.tight_layout()

**Experimental comparison — ion counting**

* Gebala, Johnson, Narlikar & Herschlag, *eLife* 2019;8:e44993 measured the ion atmosphere of nucleosomes by ion counting. Their conclusions you can test qualitatively: (i) the nucleosome keeps a strong negative electrostatic field although its net charge is roughly halved relative to free DNA; (ii) charge compensation comes mainly from **cation accumulation**, with anion exclusion contributing less; (iii) the nucleosome attracts the divalent cation (Mg²⁺) preferentially over monovalent ions. Get the measured excess-ion numbers and the salt conditions from the paper's figures/tables before overlaying — they used Na⁺/Mg²⁺ (not K⁺) and a specific nucleosome construct.
* Free-DNA benchmark for the same quantities: Bai, Greenfeld, Travers & Herschlag, *JACS* 2007, 129:14981 (ion counting on a short DNA duplex). Running the same analysis on a bare-DNA simulation gives the nucleosome/free-DNA comparison that Gebala et al. make.
* Caveat: the simulation uses a dielectric-scaled, implicit-water model with bare-charge point ions, so compare the *ordering and trends* (Mg²⁺ ≫ K⁺ per unit charge, cation-dominated neutralisation), and quote numbers with the plateau radius stated.

## 9. Local Mg²⁺ along the DNA vs. crystallographic metal sites

In [ ]:
Lb = REF["L"]; mg = "Mg" if "ploc_Mg" in RES[cond][list(RES[cond])[0]] else None
if mg:
    V_loc = 4 / 3 * np.pi * R_LOC**3
    loc = np.array([np.nanmean(prod(cond, r, "ploc_Mg"), 0) for r in RES[cond]]) * 1660.539 / V_loc   # M
    m, e = mean_sem(loc)
    fig, ax = plt.subplots(figsize=(11, 3.6))
    ax.errorbar(np.arange(Lb), m[:Lb], e[:Lb], fmt=".-", lw=.6, label="strand I phosphates")
    ax.errorbar(Lb - 1 - np.arange(Lb), m[Lb:], e[Lb:], fmt=".-", lw=.6, label="strand J (plotted at its paired bp)")
    for s in REF["mn_sites"]: ax.axvline(s["bp"], c="r", ls=":", lw=.8)
    ax.axhline(np.nanmean([np.nanmean(prod(cond, r, "bulk_M_Mg")) for r in RES[cond]]), c="k", ls="--", label="bulk Mg²⁺")
    ax.set(xlabel="bp index", ylabel=f"[Mg²⁺] within {R_LOC:.0f} Å of phosphate (M)"); ax.legend(fontsize=7); plt.tight_layout()
    print("red lines = bp coordinating crystallographic Mn²⁺ (2CV5):", sorted({s['bp'] for s in REF['mn_sites']}))
    site_vals = [m[s["strand"] * Lb + s["nt"]] for s in REF["mn_sites"]]
    print("mean local [Mg] at phosphates of Mn-coordinating nucleotides:", np.round(np.nanmean(site_vals), 3),
          "| all phosphates:", np.round(np.nanmean(m), 3), "M")

**Experimental comparison — metal binding sites**

* Crystallography: 2CV5 was crystallised with **Mn²⁺**; the 9 resolved Mn²⁺ ions sit on guanine N7/O6 (major groove) and a few protein sites (Tsunaka 2005). Davey 2002 (1KX5) likewise places Mg²⁺/Mn²⁺ on the DNA. Your CG DNA has no base atoms, so test the *coarse* statement only: is local Mg²⁺ enriched near the base-pair positions where crystal metals sit and where DNA is strongly bent/inner-gyre? Treat agreement as suggestive, not as site-specific validation (crystal contacts, Mn²⁺ ≠ Mg²⁺, 100 K, packing).
* Global level: the Mg²⁺ enrichment relative to bulk compares with ion-counting results above.

## 10. Mg²⁺ scan summary (fill `mg_mM` in `CONDITIONS` and add one condition per concentration)

In [ ]:
def summarize(cond):
    rows = {}
    for r in RES[cond]:
        d = dict(Q=prod(cond, r, "Q").mean(), wrapped_bp=prod(cond, r, "n_wrap").mean(),
                 end_dist=prod(cond, r, "end_dist").mean(), wrap_angle=np.nanmean(prod(cond, r, "wrap_angle")))
        for k in ION_RESNAMES:
            if f"gam_{k}" in RES[cond][r]:
                ip = int(np.argmin(abs(EDGES[1:] - R_PLATEAU)))
                d[f"Gamma_{k}@{R_PLATEAU:.0f}"] = np.nanmean(prod(cond, r, f"gam_{k}")[:, ip])
        rows[r] = d
    return pd.DataFrame(rows).T

summary = {c: summarize(c) for c in RES}
SUM = pd.concat({c: pd.DataFrame({"mean": s.mean(), "sem": s.std(ddof=1) / np.sqrt(len(s))}).T for c, s in summary.items()})
print(SUM.round(3))
xs = [(CONDITIONS[c]["mg_mM"], c) for c in RES if CONDITIONS[c]["mg_mM"] is not None]
if len(xs) > 1:
    xs.sort(); cols = ["Q", "wrapped_bp", "end_dist"] + [c for c in summary[xs[0][1]].columns if c.startswith("Gamma_Mg")]
    fig, ax = plt.subplots(1, len(cols), figsize=(4 * len(cols), 3.4))
    for a, col in zip(np.atleast_1d(ax), cols):
        a.errorbar([x for x, _ in xs], [summary[c][col].mean() for _, c in xs],
                   [summary[c][col].std(ddof=1) / np.sqrt(len(summary[c])) for _, c in xs], fmt="o-"); a.set(xlabel="[MgCl2] (mM)", ylabel=col)
    plt.tight_layout()

**Experimental comparison — Mg²⁺ dependence**

* Ion counting (Gebala 2019): Mg²⁺ binding per nucleosome as a function of Mg²⁺ in a monovalent background → compare Γ_Mg(c) directly (same axes).
* Chromatin compaction/self-association vs. Mg²⁺ (e.g. Schwarz & Hansen 1994 *JBC* 269:16284; Allahverdi *et al.* 2011 *NAR* 39:1680; Korolev, Nordenskiöld and co-workers): arrays fold and nucleosome core particles aggregate at low-mM Mg²⁺. A mononucleosome simulation cannot reproduce inter-nucleosome compaction, but the *Mg²⁺-dependent screening of DNA–DNA/DNA–histone electrostatics* (Γ_Mg, entry/exit-DNA distance, end unwrapping) is the single-particle precursor — state it as such.
* Keep the monovalent background fixed across the scan and use ≥ 4–5 Mg²⁺ concentrations spanning 0–10 mM (the experimental transition region).

**Not included on purpose** (no direct experimental observable): Mg²⁺ bridges, bridge–wrapping correlations, Mg²⁺ residence times. Report them only as mechanistic predictions, clearly labelled as such.

## 11. Add your own analysis

In [ ]:
# Template: copy, rename, re-run the "Run" cell (set FORCE_RECOMPUTE = True once so the cache is rebuilt).
@observable(every=1)
def obs_example(ctx, S):
    core_rg = np.sqrt(((ctx["core"] - ctx["core"].mean(0))**2).sum(1).mean())
    return dict(core_rg=core_rg)
# Then plot with:  pooled(cond, "core_rg")   /   prod(cond, rep, "core_rg")   /   rep_means(cond, "core_rg")
# ...and add an "Experimental comparison" markdown cell below it before it goes into the paper.